n Q1.1 we created a data representation, which allows us to feed multi-variate,
irregularly-sampled time series data into deep neural network sequence architectures (Q2.2,
Q2.3a). However, you might have noticed that we filled a lot of missing values using
imputation methods. Based on the work by Horn et al.
4 we might be able to avoid these
imputation steps.
Instead of modeling a sequence of time steps, we can model a sequence of measurements!
Horn et al. proposed to encode each measurement into three components: time, variable,
and value. Reprocess the data to obtain for each patient and each measurement a triplet (t,
z, v) where t is a scaled representation of time (e.g. to the range [0,1]), z is a categorical
encoding of the variable (you will have 41 categories and you could for example one-hot
encode them), and v is the scaled value observed (you can reuse the same scaling methods
already fitted in Q1.3 when preparing the time grid data. (2 pts)
Feed those sequences through a Transformer architecture, train, validate, and report test set
performance. Can you achieve the same performance or even better as with the imputed
time grid? (2 pts)


In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

# Ensure the output directory exists
os.makedirs("parquet_files", exist_ok=True)

# =====================================================================
# PHASE 1: DATA TOKENIZATION (THE TRIPLETS)
# =====================================================================
print("--- PHASE 1: Checking/Generating Tokenized Data ---")

# We only need sets "a" (train) and "c" (test) for this pipeline
for dataset_name in ["a", "c"]:  
    output_path = f"parquet_files/tokenized_triplets_set_{dataset_name}.parquet"
    
    # If the file already exists, we can skip generating it again to save time
    if os.path.exists(output_path):
        print(f"Tokenized data for set '{dataset_name}' already exists. Skipping generation.")
        continue
        
    print(f"Generating triplets for set '{dataset_name}'...")
    pathToData = f"physionet.org/files/challenge-2012/1.0.0/set-{dataset_name}" 
    outcome_path = f"physionet.org/files/challenge-2012/1.0.0/Outcomes-{dataset_name}.txt"

    all_data = []
    outcomes_df = pd.read_csv(outcome_path)[['RecordID', 'In-hospital_death']]

    for file in tqdm(os.listdir(pathToData)): 
        filepath = os.path.join(pathToData, file) 
        if not filepath.endswith(".txt"): continue
            
        record_id = int(file.replace(".txt", ""))
        dataframe = pd.read_csv(filepath) 

        # 1. TIME (t): Convert HH:MM to scaled [0, 1] range
        def parse_time(x):
            hours, mins = x.split(":")
            return int(hours) + (int(mins) / 60.0)
        
        dataframe["Time_hrs"] = dataframe["Time"].apply(parse_time)
        dataframe["t"] = (dataframe["Time_hrs"] / 48.0).clip(upper=1.0) 
        
        # 2. RENAME TO TRIPLETS
        dataframe["PatientID"] = record_id
        dataframe = dataframe.rename(columns={"Parameter": "z", "Value": "v"})
        dataframe = dataframe[dataframe["z"] != "RecordID"]
        dataframe = dataframe[["PatientID", "t", "z", "v"]]
        all_data.append(dataframe)

    full_df = pd.concat(all_data, ignore_index=True)

    # 3. VARIABLE ENCODING (z)
    full_df["z_encoded"] = pd.Categorical(full_df["z"]).codes

    # 4. MERGE OUTCOMES & SAVE
    full_df = full_df.merge(outcomes_df, left_on='PatientID', right_on='RecordID', how='left')
    full_df = full_df.drop(columns=["RecordID"])
    full_df.to_parquet(output_path, engine="pyarrow", index=False)

# =====================================================================
# PHASE 2: DATA PREPARATION & PADDING
# =====================================================================
print("\n--- PHASE 2: Loading Data & Padding Sequences ---")
train_df = pd.read_parquet("parquet_files/tokenized_triplets_set_a.parquet")
test_df = pd.read_parquet("parquet_files/tokenized_triplets_set_c.parquet")

# Scale the 'v' (Values) based ONLY on the training set
scaler = StandardScaler()
train_df['v'] = scaler.fit_transform(train_df[['v']])
test_df['v'] = scaler.transform(test_df[['v']])

def build_patient_sequences(df, max_len=500):
    grouped = df.groupby('PatientID')
    t_list, z_list, v_list, y_list, mask_list = [], [], [], [], []
    PAD_TOKEN_Z = 41 # Category 41 is reserved for empty/padded steps
    
    for _, group in grouped:
        t_seq = group['t'].values[:max_len]
        z_seq = group['z_encoded'].values[:max_len]
        v_seq = group['v'].values[:max_len]
        label = group['In-hospital_death'].iloc[0]
        
        seq_len = len(t_seq)
        
        t_padded = np.zeros(max_len, dtype=np.float32)
        z_padded = np.full(max_len, PAD_TOKEN_Z, dtype=np.int64)
        v_padded = np.zeros(max_len, dtype=np.float32)
        mask = np.ones(max_len, dtype=bool) 
        
        t_padded[:seq_len] = t_seq
        z_padded[:seq_len] = z_seq
        v_padded[:seq_len] = v_seq
        mask[:seq_len] = False 
        
        t_list.append(t_padded)
        z_list.append(z_padded)
        v_list.append(v_padded)
        y_list.append(label)
        mask_list.append(mask)
        
    return (torch.tensor(np.array(t_list)).unsqueeze(-1), 
            torch.tensor(np.array(z_list)),               
            torch.tensor(np.array(v_list)).unsqueeze(-1), 
            torch.tensor(np.array(y_list), dtype=torch.float32), 
            torch.tensor(np.array(mask_list)))            

train_t, train_z, train_v, train_y, train_mask = build_patient_sequences(train_df)
test_t, test_z, test_v, test_y, test_mask = build_patient_sequences(test_df)

class TripletDataset(Dataset):
    def __init__(self, t, z, v, y, mask):
        self.t, self.z, self.v, self.y, self.mask = t, z, v, y, mask
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return self.t[idx], self.z[idx], self.v[idx], self.y[idx], self.mask[idx]

train_loader = DataLoader(TripletDataset(train_t, train_z, train_v, train_y, train_mask), batch_size=32, shuffle=True)

# =====================================================================
# PHASE 3: THE TRIPLET TRANSFORMER MODEL
# =====================================================================
class TripletTransformer(nn.Module):
    def __init__(self, num_vars=42, d_model=64, n_heads=4, num_layers=2):
        super().__init__()
        
        self.z_emb = nn.Embedding(num_embeddings=num_vars, embedding_dim=d_model - 2, padding_idx=41)
        self.mix_layer = nn.Linear(d_model, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=128, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, t, z, v, padding_mask):
        z_embedded = self.z_emb(z) 
        x = torch.cat([t, v, z_embedded], dim=-1) 
        x = self.mix_layer(x)
        
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        
        mask_float = (~padding_mask).unsqueeze(-1).float() 
        x = x * mask_float 
        sum_x = x.sum(dim=1)
        count_x = mask_float.sum(dim=1).clamp(min=1) 
        mean_x = sum_x / count_x 
        
        return self.fc(mean_x).squeeze(-1)

# =====================================================================
# PHASE 4: TRAINING LOOP & EVALUATION
# =====================================================================
print("\n--- PHASE 4: Training Triplet Transformer ---")
model = TripletTransformer()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_t, batch_z, batch_v, batch_y, batch_mask in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_t, batch_z, batch_v, batch_mask)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

print("\n--- Final Evaluation on Test Set (C) ---")
model.eval()
with torch.no_grad():
    test_probs = model(test_t, test_z, test_v, test_mask)

probs_np = test_probs.cpu().numpy()
y_test_np = test_y.cpu().numpy()

auroc = roc_auc_score(y_test_np, probs_np)
ap = average_precision_score(y_test_np, probs_np)

print("\n=== TRIPLET TRANSFORMER PERFORMANCE ===")
print(f"AUROC Score:           {auroc:.4f}")
print(f"Average Precision (AP): {ap:.4f}")

--- PHASE 1: Checking/Generating Tokenized Data ---
Generating triplets for set 'a'...


100%|██████████| 4001/4001 [00:08<00:00, 448.87it/s]


Generating triplets for set 'c'...


100%|██████████| 4000/4000 [00:08<00:00, 457.05it/s]



--- PHASE 2: Loading Data & Padding Sequences ---

--- PHASE 4: Training Triplet Transformer ---


TypeError: Embedding.__init__() got an unexpected keyword argument 'num_classes'